# Week 04 Lab — Advanced Divide and Conquer

## Objectives
- Understand the recursive structure of divide and conquer algorithms and trace their execution.
- Implement and compare divide and conquer algorithms against brute-force approaches.

## Contents
- **A-1**: Merge Sort Tracing
- **A-2**: Finding the k-th Smallest Element (Randomized Select)
- **A-3**: Closest Pair of Points

---
## A-1: Merge Sort Tracing

Merge Sort is a classic divide and conquer algorithm:
1. **Divide**: Split the array into two halves
2. **Conquer**: Recursively sort each half
3. **Combine**: Merge the two sorted halves

Run the cell below to observe the recursive call tree of Merge Sort. Pay attention to:
- How the array is split at each level
- The order in which sub-arrays are merged
- The depth of recursion

In [ ]:
def merge_sort_trace(arr, depth=0):
    """Merge sort with recursive call visualization."""
    indent = "  " * depth
    print(f"{indent}merge_sort({arr})")

    if len(arr) <= 1:
        return arr[:]

    mid = len(arr) // 2
    left = merge_sort_trace(arr[:mid], depth + 1)
    right = merge_sort_trace(arr[mid:], depth + 1)

    merged = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1
    merged.extend(left[i:])
    merged.extend(right[j:])

    print(f"{indent}  -> merged: {merged}")
    return merged


data = [38, 27, 43, 3, 9, 82, 10]
print("=== Merge Sort Trace ===")
print()
result = merge_sort_trace(data)
print(f"\nFinal result: {result}")

### Discussion: Merge Sort Tracing

1. How many levels of recursion are there for an array of size 7?
2. At each level, how many merge operations happen?
3. What is the total number of comparisons across all merge steps?
4. If the input array were already sorted, would Merge Sort still perform the same number of operations? Why or why not?

### Exercise: Count Comparisons

Modify the merge sort below to count the total number of comparisons made during the merge steps. Fill in the `TODO` sections.

In [ ]:
def merge_sort_count(arr):
    """Merge sort that returns (sorted_array, comparison_count)."""
    if len(arr) <= 1:
        return arr[:], 0

    mid = len(arr) // 2
    left, left_count = merge_sort_count(arr[:mid])
    right, right_count = merge_sort_count(arr[mid:])

    merged = []
    comparisons = 0
    i = j = 0
    while i < len(left) and j < len(right):
        # TODO: Increment comparisons by 1 for each comparison
        comparisons += 1
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1
    merged.extend(left[i:])
    merged.extend(right[j:])

    # TODO: Return merged array and total comparison count
    # (this merge's comparisons + left side + right side)
    total = comparisons + left_count + right_count
    return merged, total


# Test with different arrays
test_arrays = [
    [38, 27, 43, 3, 9, 82, 10],
    [1, 2, 3, 4, 5, 6, 7],       # already sorted
    [7, 6, 5, 4, 3, 2, 1],       # reverse sorted
]

for arr in test_arrays:
    sorted_arr, count = merge_sort_count(arr)
    print(f"Input: {arr}")
    print(f"Sorted: {sorted_arr}, Comparisons: {count}")
    print()

---
## A-2: Finding the k-th Smallest Element

**Randomized Select** finds the k-th smallest element in expected O(n) time.

Key idea:
- Uses the **partition** step from Quick Sort
- After partitioning, the pivot is in its correct sorted position
- We only recurse into the side that contains our target index

This is much faster than sorting the entire array (O(n log n)) when we only need one element.

In [ ]:
import random


def partition(arr, left, right, pivot_idx):
    """Partition arr[left..right] around the pivot element.
    
    Returns the final index of the pivot after partitioning.
    """
    pivot = arr[pivot_idx]
    arr[pivot_idx], arr[right] = arr[right], arr[pivot_idx]
    store = left
    for i in range(left, right):
        if arr[i] < pivot:
            arr[i], arr[store] = arr[store], arr[i]
            store += 1
    arr[store], arr[right] = arr[right], arr[store]
    return store


def randomized_select(arr, left, right, k):
    """Find the k-th smallest element (0-indexed) in arr[left..right]."""
    if left == right:
        return arr[left]

    pivot_idx = random.randint(left, right)
    pivot_idx = partition(arr, left, right, pivot_idx)

    if k == pivot_idx:
        return arr[k]
    elif k < pivot_idx:
        return randomized_select(arr, left, pivot_idx - 1, k)
    else:
        return randomized_select(arr, pivot_idx + 1, right, k)


def kth_smallest(arr, k):
    """Find k-th smallest (1-indexed) element."""
    a = arr[:]
    return randomized_select(a, 0, len(a) - 1, k - 1)


# Demo: find every k-th smallest element
data = [7, 10, 4, 3, 20, 15, 8]
print(f"Array: {data}")
print(f"Sorted: {sorted(data)}")
for k in range(1, len(data) + 1):
    result = kth_smallest(data, k)
    print(f"  {k}-th smallest: {result}")

### Performance Comparison: Randomized Select vs. Sort + Index

Let's compare the performance of Randomized Select (O(n) average) against sorting the entire array (O(n log n)) and then indexing.

In [ ]:
import time

n = 1_000_000
big_data = [random.randint(1, 10**9) for _ in range(n)]

# Randomized Select
start = time.perf_counter()
result1 = kth_smallest(big_data, n // 2)
t1 = time.perf_counter() - start

# Sort + Index
start = time.perf_counter()
result2 = sorted(big_data)[n // 2 - 1]
t2 = time.perf_counter() - start

print(f"N = {n:,}, finding median:")
print(f"  Randomized Select: {t1:.4f}s (result = {result1})")
print(f"  Sort + index:      {t2:.4f}s (result = {result2})")
print(f"  Speedup:           {t2/t1:.1f}x")

### Exercise: Deterministic Select (Median of Medians)

The Randomized Select algorithm has O(n) *expected* time, but O(n^2) worst case.
The **Median of Medians** algorithm guarantees O(n) worst-case by choosing a good pivot.

**TODO**: Implement the `median_of_medians` function below that:
1. Divides the array into groups of 5
2. Finds the median of each group
3. Recursively finds the median of those medians
4. Uses that as the pivot for partitioning

In [ ]:
def median_of_medians(arr, k):
    """Find the k-th smallest element (0-indexed) using Median of Medians.
    
    Guarantees O(n) worst-case time complexity.
    
    Args:
        arr: list of numbers
        k: 0-indexed position of the element to find
        
    Returns:
        The k-th smallest element
    """
    if len(arr) <= 5:
        return sorted(arr)[k]

    # TODO: Step 1 - Split arr into groups of 5 and find median of each group
    # Hint: medians = [sorted(arr[i:i+5])[len(arr[i:i+5])//2] for i in range(0, len(arr), 5)]
    medians = []  # Replace with your implementation

    # TODO: Step 2 - Find the median of medians (recursively)
    # Hint: pivot = median_of_medians(medians, len(medians) // 2)
    pivot = None  # Replace with your implementation

    # TODO: Step 3 - Partition into three groups: less than, equal to, greater than pivot
    low = [x for x in arr if x < pivot]
    equal = [x for x in arr if x == pivot]
    high = [x for x in arr if x > pivot]

    # TODO: Step 4 - Recurse into the correct partition
    if k < len(low):
        return median_of_medians(low, k)
    elif k < len(low) + len(equal):
        return pivot
    else:
        return median_of_medians(high, k - len(low) - len(equal))


# Test your implementation
test_data = [7, 10, 4, 3, 20, 15, 8, 1, 17, 12, 6, 11]
print(f"Array: {test_data}")
print(f"Sorted: {sorted(test_data)}")
for k in range(len(test_data)):
    result = median_of_medians(test_data[:], k)
    expected = sorted(test_data)[k]
    status = "OK" if result == expected else f"FAIL (got {result})"
    print(f"  k={k}: {expected} ... {status}")

---
## A-3: Closest Pair of Points

Given n points in 2D space, find the pair of points with the smallest Euclidean distance.

**Approaches**:
- **Brute force**: Check all pairs -- O(n^2)
- **Divide and conquer**: Split points by x-coordinate, solve each half, check the strip -- O(n log n)

The D&C approach is significantly faster for large inputs.

In [ ]:
import math


def dist(p1, p2):
    """Euclidean distance between two 2D points."""
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)


def closest_pair_bruteforce(points):
    """O(n^2): Check all pairs."""
    n = len(points)
    min_dist = float('inf')
    pair = None
    for i in range(n):
        for j in range(i + 1, n):
            d = dist(points[i], points[j])
            if d < min_dist:
                min_dist = d
                pair = (points[i], points[j])
    return min_dist, pair


def closest_pair_dc(points):
    """O(n log n): Divide and conquer approach."""
    points_sorted = sorted(points, key=lambda p: p[0])
    return _closest_dc(points_sorted)


def _closest_dc(pts):
    n = len(pts)
    if n <= 3:
        return closest_pair_bruteforce(pts)

    mid = n // 2
    mid_x = pts[mid][0]
    left_result = _closest_dc(pts[:mid])
    right_result = _closest_dc(pts[mid:])

    d = min(left_result[0], right_result[0])
    best = left_result if left_result[0] <= right_result[0] else right_result

    # Check the strip around the dividing line
    strip = [p for p in pts if abs(p[0] - mid_x) < d]
    strip.sort(key=lambda p: p[1])

    for i in range(len(strip)):
        j = i + 1
        while j < len(strip) and strip[j][1] - strip[i][1] < d:
            dd = dist(strip[i], strip[j])
            if dd < d:
                d = dd
                best = (dd, (strip[i], strip[j]))
            j += 1

    return best


# Small example
points = [(2, 3), (12, 30), (40, 50), (5, 1), (12, 10), (3, 4)]
d1, pair1 = closest_pair_bruteforce(points)
d2, pair2 = closest_pair_dc(points)
print(f"Points: {points}")
print(f"Brute force: dist={d1:.4f}, pair={pair1}")
print(f"D&C:         dist={d2:.4f}, pair={pair2}")

### Performance Comparison: Brute Force vs. Divide and Conquer

Run the cell below to compare the two approaches on increasingly large inputs.

In [ ]:
import time

print(f"{'N':>6}  {'Brute Force':>12}  {'D&C':>12}  {'Speedup':>8}")
print("-" * 46)

for n in [100, 500, 1000, 3000, 5000]:
    pts = [(random.uniform(0, 10000), random.uniform(0, 10000)) for _ in range(n)]

    start = time.perf_counter()
    closest_pair_bruteforce(pts)
    t1 = time.perf_counter() - start

    start = time.perf_counter()
    closest_pair_dc(pts)
    t2 = time.perf_counter() - start

    speedup = t1 / t2 if t2 > 0 else float('inf')
    print(f"{n:>6}  {t1:>11.4f}s  {t2:>11.4f}s  {speedup:>7.1f}x")

### Discussion: Closest Pair

1. Why do we only need to check at most 6-8 points in the strip for each point?
2. How does the speedup ratio change as N grows? Does this match the theoretical O(n^2) vs O(n log n) prediction?
3. What would happen if all points had the same x-coordinate? How would this affect the divide and conquer approach?

### Exercise: Visualizing the Closest Pair

**TODO**: Complete the visualization function below to plot the points and highlight the closest pair. This uses `matplotlib` -- install it with `pip install matplotlib` if needed.

In [ ]:
try:
    import matplotlib.pyplot as plt

    def visualize_closest_pair(points):
        """Plot points and highlight the closest pair."""
        d, pair = closest_pair_dc(points)

        xs = [p[0] for p in points]
        ys = [p[1] for p in points]

        plt.figure(figsize=(8, 6))
        plt.scatter(xs, ys, c='blue', s=20, alpha=0.6, label='Points')

        # TODO: Highlight the closest pair
        # Hint: Plot the two points in red and draw a line between them
        if pair:
            p1, p2 = pair
            plt.scatter([p1[0], p2[0]], [p1[1], p2[1]], c='red', s=100, zorder=5,
                        label=f'Closest pair (d={d:.2f})')
            plt.plot([p1[0], p2[0]], [p1[1], p2[1]], 'r-', linewidth=2)

        plt.title(f'Closest Pair of Points (n={len(points)})')
        plt.xlabel('x')
        plt.ylabel('y')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

    # Generate random points and visualize
    random_points = [(random.uniform(0, 100), random.uniform(0, 100)) for _ in range(50)]
    visualize_closest_pair(random_points)

except ImportError:
    print("matplotlib is not installed. Run: pip install matplotlib")
    print("Skipping visualization.")

---
## Summary

| Algorithm | Approach | Time Complexity | Key Idea |
|-----------|----------|----------------|----------|
| Merge Sort | Divide & Conquer | O(n log n) | Split, sort halves, merge |
| Randomized Select | Divide & Conquer | O(n) average | Partition around random pivot, recurse on one side |
| Closest Pair (Brute Force) | Exhaustive | O(n^2) | Check all pairs |
| Closest Pair (D&C) | Divide & Conquer | O(n log n) | Split by x, check strip |

### Key Takeaways
- Divide and conquer reduces problem size exponentially by splitting into subproblems.
- Not all problems require solving both subproblems (Randomized Select only recurses into one side).
- The "combine" step is often the most critical part -- the strip check in Closest Pair is what makes O(n log n) possible.